# Week 5: Semantic Search with Embeddings

This notebook compares semantic retrieval with BM25 keyword matching over cleaned real-estate listing remarks. Run the build commands first to create the local corpus, embeddings, and FAISS index.

## 1. Build local artifacts

With Docker/MySQL running, run these from the project terminal:

```bash
python scripts/build_semantic_corpus.py
python scripts/build_embeddings.py
python scripts/build_faiss_index.py
python scripts/benchmark_semantic_search.py
python scripts/compare_search.py --top-k 5
```

The first command exports and cleans every eligible listing; MiniLM then creates 384-dimensional vectors, and FAISS writes a cosine-similarity index. The initial model download and full embedding build can take time on a CPU-only laptop.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.bm25_search import BM25Searcher
from scripts.semantic_search import SemanticSearcher
import pandas as pd

METADATA_PATH = ROOT / "data/models/listing_metadata.csv"
INDEX_PATH = ROOT / "data/models/listings.faiss"

metadata = pd.read_csv(METADATA_PATH)
semantic = SemanticSearcher.load(INDEX_PATH, METADATA_PATH)
bm25 = BM25Searcher(metadata)
print(f"Loaded {len(metadata):,} listings; FAISS dimensions: {semantic.index.d}")

Loaded 10,000 listings; FAISS dimensions: 384


## 2. Compare semantic and keyword retrieval

Semantic search can retrieve related wording such as *expansive private yard* for a query about a *large backyard*. BM25 is the lexical baseline: it scores overlapping words.

In [2]:
query = "quiet family friendly home with outdoor space"

semantic_results = semantic.search(query, top_k=5)
bm25_results = bm25.search(query, top_k=5)

display(semantic_results[["rank", "listing_id", "address", "city", "price", "similarity_score", "remarks_cleaned"]])
display(bm25_results[["rank", "listing_id", "address", "city", "price", "bm25_score", "remarks_cleaned"]])

/Users/dengyingxin/Desktop/IDX Exchange/NLP intern/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


,rank,listing_id,address,city,price,similarity_score,remarks_cleaned
0,1,1146411410,8267 San Bernardino,Rancho Cucamonga,1225000,0.646280,Welcome to a home that brings the family close...
1,2,1155349515,1625 Sheldon,Modesto,457000,0.638815,Gorgeous single story home with two-car garage...
2,3,1114552303,24931 Heil Drive,Moreno Valley,597000,0.632026,Discover the perfect blend of modern updates a...
3,4,1155306563,5129 Toyon,Paradise,399000,0.629150,Nestled among the trees in a quiet neighborhoo...
4,5,1155038384,1672 Carmel W,Upland,510000,0.625504,One-Level 2 bedrooms and 2 full bathrooms in a...


,rank,listing_id,address,city,price,bm25_score,remarks_cleaned
0,1,1156934366,3909 Reche Rd SPC 116,Fallbrook,199900,17.767998,"Welcome to 3909 Reche Rd, Space 116 - a charmi..."
1,2,1147124005,1350 Mistletoe Dr,Beaumont,550000,17.667416,"Welcome to 1350 Misteltoe Drive, a charming si..."
2,3,1156037226,3525 Cromwell Court,Perris,560000,17.244301,Tucked away on a cul-de-sac in this growing co...
3,4,1145948172,1124 S Cajon Avenue,West Covina,968000,16.741815,SPACIOUS RANCH RETREAT WITH ENDLESS POTENTIAL ...
4,5,1151668863,62915 Loftwood Street,Desert Hot Springs,440000,16.712538,Welcome Home!! Skyborne is one of the newer de...


## 3. Latency benchmark

The target is under 100 ms per query for 10,000 listings after the model is loaded. The measurement includes query embedding and FAISS retrieval.

In [3]:
benchmark_queries = [
    "quiet family home", "modern kitchen", "private backyard", "ocean view condo",
    "move in ready property", "home office", "pool and spa", "near restaurants",
    "newly renovated home", "resort style amenities",
]
# One warm-up query loads the model and avoids counting one-time setup cost.
semantic.search(benchmark_queries[0], top_k=10)
semantic.benchmark(benchmark_queries, top_k=10)

{'query_count': 10, 'average_ms': 17.24, 'median_ms': 15.88, 'p95_ms': 25.06}

## 4. Relevance evaluation

`scripts/compare_search.py` creates `data/processed/week5_relevance_review.csv`: 100 balanced query-result pairs (10 queries × 5 results × 2 methods). Set `relevant` to `yes` or `no`, explain ambiguous cases in `notes`, and compare the methods using the same Precision@5 evaluation protocol.

### Full-corpus evaluation status

The completed local corpus contains **54,838** cleaned listing remarks. The comparison uses 100 balanced result pairs: 10 queries × 5 results × 2 methods. The labels currently included are clearly marked as **assistant-proposed review labels**, not independent human gold labels; a human reviewer should revise them before making a final relevance claim.


In [ ]:
review_path = ROOT / 'data/processed/week5_relevance_review.csv'
review = pd.read_csv(review_path)

if review['relevant'].fillna('').astype(str).str.strip().eq('').any():
    raise ValueError('Complete the relevant column before calculating Precision@5.')

evaluation = (
    review.assign(is_relevant=review['relevant'].eq('yes'))
    .groupby('search_method', as_index=False)
    .agg(result_pairs=('is_relevant', 'size'), relevant_pairs=('is_relevant', 'sum'))
)
evaluation['precision_at_5'] = evaluation['relevant_pairs'] / evaluation['result_pairs']
evaluation
